# Lab 8 — Quantum Convolutional Neural Network (QCNN)

## Objectifs
- Implémenter une convolution quantique et un pooling quantique
- Construire une architecture QCNN hiérarchique
- Comparer avec un CNN classique
- Analyser l'impact de la profondeur du circuit

In [ ]:
import pennylane as qml
from pennylane import qnode
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import time

## Partie A — Convolution quantique

La convolution quantique est l'équivalent des filtres convolutifs classiques : elle applique des rotations paramétrées et des portes CNOT sur des qubits adjacents, en partageant les paramètres comme un filtre classique partage ses poids.

In [ ]:
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

def quantum_conv(weights, wires):
    """Couche de convolution quantique sur 2 qubits adjacents.
    Applique des rotations paramétrées + CNOT (partage de paramètres).
    """
    n_wires = len(wires)
    for i in range(n_wires):
        qml.RY(weights[0], wires=wires[i])
    for i in range(n_wires - 1):
        qml.CNOT(wires=[wires[i], wires[i + 1]])
    for i in range(n_wires):
        qml.RY(weights[1], wires=wires[i])
    for i in range(n_wires - 1):
        qml.CNOT(wires=[wires[i], wires[i + 1]])

print("Couche de convolution quantique définie (2 qubits, paramètres partagés)")
print("Structure : RY → CNOT → RY → CNOT")

dev_conv = qml.device("default.qubit", wires=2)

@qml.qnode(dev_conv)
def conv_demo(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(2))
    quantum_conv(weights, wires=range(2))
    return [qml.expval(qml.PauliZ(i)) for i in range(2)]

print(qml.draw(conv_demo)(np.array([0.5, 0.3]), np.array([[0.1, 0.2], [0.3, 0.4]])))

## Partie B — Pooling quantique

Le pooling quantique réduit le nombre de qubits actifs de moitié. Il s'inspire du max-pooling classique : la mesure d'un qubit source conditionne une rotation sur le qubit cible, compressant l'information.

In [ ]:
def quantum_pool(source_wire, target_wire):
    """Couche de pooling quantique.
    Mesure le qubit source et effectue une rotation conditionnelle sur la cible.
    Réduit le nombre de qubits actifs de moitié.
    """
    qml.CNOT(wires=[source_wire, target_wire])
    qml.RY(np.pi / 4, wires=target_wire)

print("Couche de pooling quantique définie")
print("Opération : CNOT(source, target) → RY(π/4, target)")
print("Réduction : 4 qubits → 2 qubits → 1 qubit")

## Partie C — Architecture QCNN complète

L'architecture QCNN combine convolutions et pooling de manière hiérarchique :
1. **Conv(0,1), Conv(2,3)** : deux convolutions en parallèle
2. **Pool(0→1), Pool(2→3)** : réduction de 4 à 2 qubits actifs
3. **Conv(1,3)** : convolution sur les qubits restants
4. **Pool(1→3)** : réduction finale à 1 qubit
5. **Mesure Z(3)** : classification binaire

In [ ]:
@qml.qnode(dev, interface="torch")
def qcnn_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))

    quantum_conv(weights[0], wires=[0, 1])
    quantum_conv(weights[1], wires=[2, 3])

    quantum_pool(0, 1)
    quantum_pool(2, 3)

    quantum_conv(weights[2], wires=[1, 3])

    quantum_pool(1, 3)

    return qml.expval(qml.PauliZ(3))

weight_shapes = {
    "weights": [(2, 2), (2, 2), (2, 2)]
}

qlayer = qml.qnn.TorchLayer(qcnn_circuit, weight_shapes=weight_shapes)

class QCNNModel(nn.Module):
    def __init__(self, qlayer):
        super().__init__()
        self.qlayer = qlayer

    def forward(self, x):
        return self.qlayer(x)

model_qcnn = QCNNModel(qlayer)

n_params = sum(p.numel() for p in model_qcnn.parameters() if p.requires_grad)
print(f"Architecture QCNN : {n_qubits} qubits")
print(f"Paramètres entraînables : {n_params}")
print(f"\nFlux : AngleEmbedding → Conv(0,1) Conv(2,3) → Pool → Conv(1,3) → Pool → Z(3)")
print(f"\nCircuit QCNN :")
print(qml.draw(qcnn_circuit)(np.random.rand(n_qubits), [np.random.rand(2, 2) for _ in range(3)]))

In [ ]:
X, y = make_classification(
    n_samples=400,
    n_features=n_qubits,
    n_informative=4,
    n_redundant=0,
    n_classes=2,
    random_state=42
)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X = scaler.fit_transform(X)

y = y.astype(np.float64) * 2 - 1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

print(f"Dataset synthétique : {X.shape[0]} échantillons, {X.shape[1]} features")
print(f"Train : {len(X_train)}, Test : {len(X_test)}")
print(f"Distribution des classes : {np.unique(y, return_counts=True)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', alpha=0.6, s=10)
axes[0].set_title('Features 0 vs 1 (train)')
axes[0].set_xlabel('Feature 0')
axes[0].set_ylabel('Feature 1')
axes[1].scatter(X_train[:, 2], X_train[:, 3], c=y_train, cmap='coolwarm', alpha=0.6, s=10)
axes[1].set_title('Features 2 vs 3 (train)')
axes[1].set_xlabel('Feature 2')
axes[1].set_ylabel('Feature 3')
plt.tight_layout()
plt.show()

## Partie D — Entraînement du QCNN

On entraîne le QCNN avec l'optimiseur Adam et la fonction de perte MSE. Le circuit quantique est simulé sur CPU, ce qui rend l'entraînement plus lent qu'un réseau classique équivalent.

In [ ]:
n_epochs = 50
lr = 0.01

criterion = nn.MSELoss()
optimizer = optim.Adam(model_qcnn.parameters(), lr=lr)

train_losses = []
test_accuracies = []

start_time_qcnn = time.time()

for epoch in range(n_epochs):
    model_qcnn.train()
    optimizer.zero_grad()
    predictions = model_qcnn(X_train_t)
    loss = criterion(predictions, y_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    model_qcnn.eval()
    with torch.no_grad():
        test_preds = model_qcnn(X_test_t)
        test_preds_binary = (test_preds > 0).float()
        y_test_binary = (y_test_t > 0).float()
        acc = accuracy_score(y_test_binary.numpy(), test_preds_binary.numpy())
        test_accuracies.append(acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} — Loss: {loss.item():.4f} — Acc: {acc:.4f}")

time_qcnn = time.time() - start_time_qcnn
print(f"\nAccuracy finale QCNN : {test_accuracies[-1]:.4f}")
print(f"Temps d'entraînement QCNN : {time_qcnn:.2f}s")

In [ ]:
# Matrice de confusion du QCNN
model_qcnn.eval()
with torch.no_grad():
    all_preds = model_qcnn(X_test_t)
    preds_binary = (all_preds > 0).float().numpy()
    true_binary = (y_test_t > 0).float().numpy()

cm_qcnn = confusion_matrix(true_binary, preds_binary)
print("Matrice de confusion QCNN :")
print(cm_qcnn)
print(f"\nAccuracy par classe :")
for i, cls in enumerate(['Classe -1', 'Classe +1']):
    class_acc = cm_qcnn[i, i] / cm_qcnn[i, :].sum()
    print(f"  {cls}: {class_acc:.4f}")

## Partie E — Comparaison avec CNN classique

On construit un CNN classique équivalent en profondeur pour comparer les performances, le nombre de paramètres et le temps d'entraînement.

In [ ]:
class ClassicalCNN(nn.Module):
    """CNN classique équivalent en profondeur au QCNN.
    Conv1D(4,4) → Pool → Conv1D(4,2) → Pool → Linear(2,1)
    """
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 4, kernel_size=2, padding=1)
        self.pool1 = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(4, 2, kernel_size=2, padding=1)
        self.pool2 = nn.MaxPool1d(2)
        self.fc = nn.Linear(2, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x))
        x = self.pool1(x)
        x = self.relu(self.conv2(x))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x.squeeze()

model_cnn = ClassicalCNN()

n_params_cnn = sum(p.numel() for p in model_cnn.parameters())
n_params_qcnn = sum(p.numel() for p in model_qcnn.parameters())

print(f"Paramètres QCNN : {n_params_qcnn}")
print(f"Paramètres CNN classique : {n_params_cnn}")
print(f"\nArchitecture CNN classique :")
print(model_cnn)

In [ ]:
criterion_cnn = nn.MSELoss()
optimizer_cnn = optim.Adam(model_cnn.parameters(), lr=lr)

cnn_losses = []
cnn_accuracies = []

start_time_cnn = time.time()

for epoch in range(n_epochs):
    model_cnn.train()
    optimizer_cnn.zero_grad()
    preds = model_cnn(X_train_t)
    loss = criterion_cnn(preds, y_train_t)
    loss.backward()
    optimizer_cnn.step()
    cnn_losses.append(loss.item())

    model_cnn.eval()
    with torch.no_grad():
        test_preds = model_cnn(X_test_t)
        test_preds_binary = (test_preds > 0).float()
        y_test_binary = (y_test_t > 0).float()
        acc = accuracy_score(y_test_binary.numpy(), test_preds_binary.numpy())
        cnn_accuracies.append(acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} — Loss: {loss.item():.4f} — Acc: {acc:.4f}")

time_cnn = time.time() - start_time_cnn
print(f"\nAccuracy finale CNN classique : {cnn_accuracies[-1]:.4f}")
print(f"Temps d'entraînement CNN : {time_cnn:.2f}s")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(train_losses, label="QCNN", color="purple", linewidth=2)
axes[0].plot(cnn_losses, label="CNN classique", color="blue", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Courbe de perte")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(test_accuracies, label="QCNN", color="purple", linewidth=2)
axes[1].plot(cnn_accuracies, label="CNN classique", color="blue", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Courbe d'accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

metrics = ["Accuracy", "Paramètres"]
qcnn_vals = [test_accuracies[-1], n_params_qcnn]
cnn_vals = [cnn_accuracies[-1], n_params_cnn]
x = np.arange(len(metrics))
width = 0.35
axes[2].bar(x - width/2, qcnn_vals, width, label="QCNN", color="purple", alpha=0.8)
axes[2].bar(x + width/2, cnn_vals, width, label="CNN classique", color="blue", alpha=0.8)
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics)
axes[2].set_title("Comparaison des métriques")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 55)
print(f"{'Métrique':<25} {'QCNN':<15} {'CNN':<15}")
print("=" * 55)
print(f"{'Accuracy test':<25} {test_accuracies[-1]:<15.4f} {cnn_accuracies[-1]:<15.4f}")
print(f"{'Paramètres':<25} {n_params_qcnn:<15} {n_params_cnn:<15}")
print(f"{'Temps (s)':<25} {time_qcnn:<15.2f} {time_cnn:<15.2f}")
print(f"{'Ratio temps (Q/C)':<25} {time_qcnn/time_cnn:<15.2f}")
print("=" * 55)

delta = cnn_accuracies[-1] - test_accuracies[-1]
if delta > 0:
    print(f"\n→ Le CNN classique surpasse le QCNN de {delta:.4f} en accuracy.")
    print(f"  Cependant, le QCNN utilise {n_params_qcnn} paramètres contre {n_params_cnn} pour le CNN.")
else:
    print(f"\n→ Le QCNN surpasse le CNN classique de {-delta:.4f} en accuracy")
    print(f"  avec seulement {n_params_qcnn} paramètres entraînables !")

## Exercices

### Exercice 1 — Profondeur du QCNN
Augmenter la profondeur du QCNN en ajoutant une couche de convolution et pooling supplémentaire. Pour cela, passer de 4 qubits à 8 qubits et observer l'impact sur l'accuracy.

```python
n_qubits_8 = 8
# Construire : Conv(0,1) Conv(2,3) Conv(4,5) Conv(6,7)
#            → Pool(0→1) Pool(2→3) Pool(4→5) Pool(6→7)
#            → Conv(1,3) Conv(5,7) → Pool(1→3) Pool(5→7)
#            → Conv(3,7) → Pool(3→7) → Z(7)
```

### Exercice 2 — Encodage IQP
Remplacer l'AngleEmbedding par un encodage IQP et comparer les performances :

```python
@qml.qnode(dev, interface="torch")
def qcnn_iqp(inputs, weights):
    qml.IQPEmbedding(inputs, wires=range(n_qubits), n_repeats=2)
    # ... même structure conv/pool
```

### Exercice 3 — Comparaison des paramètres
Compter et comparer le nombre de paramètres pour différentes architectures :
- QCNN avec 4 qubits (actuel)
- QCNN avec 8 qubits
- CNN classique équivalent
- MLP classique équivalent

### Exercice 4 — Dataset réel
Tester le QCNN sur un dataset réel (par exemple, les données Iris reduites à 4 features via PCA) et comparer avec le dataset synthétique.

```python
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
iris = load_iris()
X_iris = PCA(n_components=4).fit_transform(iris.data)
```

### Exercice 5 — Bruit simulé
Ajouter du bruit dépolarisant au simulateur et observer comment l'accuracy du QCNN dégrade :

```python
dev_noisy = qml.device('default.mixed', wires=n_qubits)
# Ajouter qml.depolarizing_channel(p=0.01, wires=i) après chaque couche
```